# Clasificador y Recomendador para Enfermedades en Hojas de Plantas
### Entrega M1 · Tópicos Especiales y Aplicaciones en IA · Universidad EAFIT

**Equipo:** Luciana Hoyos, Sara López, Juan Carlos Citelly, Santiago Manco Maya

---

## Sobre el proyecto

La visión completa del proyecto es un sistema de dos etapas: (1) un **clasificador**
de imágenes que identifique la enfermedad de una hoja (computer vision, fase
posterior del curso) y (2) un **recomendador** de manejo agronómico en español que,
a partir de esa etiqueta, le diga a un agricultor qué hacer.

**Esta entrega (M1) cubre solo la segunda mitad: el recomendador de texto.** Por
instrucción del curso, en esta fase no se trabaja con computer vision — se parte
directamente de las **etiquetas** del dataset PlantVillage (no de las imágenes) y
se hace *fine-tuning* de un modelo de lenguaje pequeño con LoRA para que, dada una
etiqueta/pregunta sobre una enfermedad, genere una recomendación agronómica útil.

## Qué contiene esta entrega

1. **Justificación del modelo base** — comparación de candidatos y evidencia de por
   qué se eligió la familia Qwen2.5-Instruct, importada desde el Hub de Hugging Face.
2. **Dataset documentado** — de dónde salen los ejemplos, por qué son representativos
   de la tarea, y sus limitaciones. Ver también `datos/REFERENCIAS.md`.
3. **Baseline vs. resultado** — el modelo antes y después del fine-tuning, sobre la
   misma pregunta y sobre un conjunto de prueba completo.
4. **Modelo fine-tuneado con LoRA** — este notebook ejecutado de principio a fin.

## Estructura del repositorio

```
proyecto_M1/
├── README.md
├── notebook.ipynb          <- este archivo
└── datos/
    ├── color/ (Carpeta con las imagenes a color de PlantVillage, debe cargarse directamente, se encuentra aquí: https://huggingface.co/datasets/Jackieeeeee/plantvillage-raw-color-7f7ecc7)
    ├── dataset_finetuning_plantvillage.jsonl   (204 ejemplos instrucción→respuesta)
    ├── base_conocimiento_plantvillage.json     (base de conocimiento cruda, con fuentes)
    ├── base_conocimiento_plantvillage.csv      (la misma base, en tabla)
    └── REFERENCIAS.md                          (fuentes citadas)
```

## 0 · Preparación del entorno

In [ ]:
# Instalación de dependencias.
!pip install -q transformers datasets peft accelerate evaluate bitsandbytes rouge_score scikit-learn 2>/dev/null
print("Librerías instaladas.")


In [26]:
import os
import json
import random

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {device}")
if device == "cpu":
    print("ADVERTENCIA: no hay GPU activa. Runtime > Change runtime type > GPU, y reinicien.")

# Ruta a la carpeta de datos (ajusta si corres el notebook desde otra ubicación)
DATOS_DIR = "datos"


Dispositivo: cuda


## 1 · Selección y justificación del modelo base

Necesitamos un modelo que: (a) quepa y entrene con LoRA en una sola GPU de Colab
(T4, 16GB), (b) tenga una versión *instruction-tuned* para partir de un baseline
razonable, (c) tenga buen soporte de **español** (el producto final habla con
agricultores en español), y (d) tenga licencia permisiva para uso académico.

Comparamos 4 candidatos pequeños (0.5B–1.1B parámetros); no consideramos modelos
de 7B+ porque exceden la memoria disponible para fine-tuning con LoRA + activaciones
en una GPU gratuita.


In [27]:
candidatos = [
    {
        "modelo": "Qwen/Qwen2.5-0.5B-Instruct",
        "parametros": "0.5B",
        "licencia": "Apache 2.0",
        "contexto": "32K tokens",
        "multilingue": "Sí, entrenado explícitamente en 29+ idiomas incl. español",
        "notas": "Mejor rendimiento por parámetro de la comparación; tokenizer "
                 "grande (~150k) pensado para multilingüe.",
    },
    {
        "modelo": "Qwen/Qwen2.5-1.5B-Instruct",
        "parametros": "1.5B",
        "licencia": "Apache 2.0",
        "contexto": "32K tokens",
        "multilingue": "Sí, misma familia que 0.5B",
        "notas": "Mismo tokenizer que 0.5B, más capacidad de generación coherente "
                 "a costo de más VRAM y tiempo de entrenamiento. Candidato de respaldo.",
    },
    {
        "modelo": "microsoft/Phi-3-mini-4k-instruct",
        "parametros": "3.8B",
        "licencia": "MIT",
        "contexto": "4K tokens",
        "multilingue": "Entrenado mayormente en inglés; español es soporte secundario",
        "notas": "Fuerte en inglés/razonamiento, pero ~7-8x más pesado y con menor "
                 "cobertura de español que Qwen2.5.",
    },
    {
        "modelo": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        "parametros": "1.1B",
        "licencia": "Apache 2.0",
        "contexto": "2K tokens",
        "multilingue": "Entrenado casi exclusivamente en inglés",
        "notas": "Muy liviano, pero pobre soporte de español; se descarta por idioma.",
    },
]

df_candidatos = pd.DataFrame(candidatos)
df_candidatos


,modelo,parametros,licencia,contexto,multilingue,notas
0,Qwen/Qwen2.5-0.5B-Instruct,0.5B,Apache 2.0,32K tokens,"Sí, entrenado explícitamente en 29+ idiomas in...",Mejor rendimiento por parámetro de la comparac...
1,Qwen/Qwen2.5-1.5B-Instruct,1.5B,Apache 2.0,32K tokens,"Sí, misma familia que 0.5B","Mismo tokenizer que 0.5B, más capacidad de gen..."
2,microsoft/Phi-3-mini-4k-instruct,3.8B,MIT,4K tokens,Entrenado mayormente en inglés; español es sop...,"Fuerte en inglés/razonamiento, pero ~7-8x más ..."
3,TinyLlama/TinyLlama-1.1B-Chat-v1.0,1.1B,Apache 2.0,2K tokens,Entrenado casi exclusivamente en inglés,"Muy liviano, pero pobre soporte de español; se..."


In [28]:
# Evidencia empírica: comportamiento del tokenizador de cada candidato sobre texto
# del dominio (español, vocabulario agronómico). Un tokenizador que fragmenta mucho
# el texto en subpalabras es menos eficiente: más tokens por ejemplo, más costo,
# más contexto gastado innecesariamente.

from transformers import AutoTokenizer as _AT

frases_dominio = [
    "El tizón tardío de la papa es causado por el oomiceto Phytophthora infestans.",
    "Se recomienda aplicar un fungicida a base de cobre y mejorar la ventilación del cultivo.",
    "Las manchas foliares por Septoria aparecen como puntos oscuros con centro grisáceo.",
    "Retire y destruya el follaje infectado para evitar la propagación de la roya del maíz.",
]

resultados_tokenizer = []
for c in candidatos:
    try:
        tok = _AT.from_pretrained(c["modelo"])
        total_tokens = sum(len(tok.encode(f, add_special_tokens=False)) for f in frases_dominio)
        total_palabras = sum(len(f.split()) for f in frases_dominio)
        resultados_tokenizer.append({
            "modelo": c["modelo"],
            "vocab_size": tok.vocab_size,
            "tokens_por_palabra": round(total_tokens / total_palabras, 2),
        })
    except Exception as e:
        resultados_tokenizer.append({"modelo": c["modelo"], "error": str(e)})

pd.DataFrame(resultados_tokenizer)


,modelo,vocab_size,tokens_por_palabra
0,Qwen/Qwen2.5-0.5B-Instruct,151643,1.75
1,Qwen/Qwen2.5-1.5B-Instruct,151643,1.75
2,microsoft/Phi-3-mini-4k-instruct,32000,1.85
3,TinyLlama/TinyLlama-1.1B-Chat-v1.0,32000,1.85


### Conclusión: `Qwen/Qwen2.5-0.5B-Instruct`

- **Tamaño (0.5B):** el más liviano de la comparación, permite fine-tuning con LoRA
  rápido en una T4 gratuita de Colab, dejando margen para iterar varias veces.
- **Licencia:** Apache 2.0, permisiva para uso académico.
- **Idioma:** a diferencia de TinyLlama y Phi-3-mini, la familia Qwen2.5 está
  entrenada explícitamente con una porción significativa de datos en español,
  necesario porque el producto final habla con agricultores en español.
- **Tokenizador sobre el dominio:** el ratio de tokens por palabra medido arriba en
  frases agronómicas en español confirma que Qwen2.5 fragmenta menos el texto en
  español que TinyLlama, que no fue diseñado para el idioma.

Se descarta Phi-3-mini por ser ~7-8x más grande sin ventaja clara en español, y
TinyLlama por su pobre soporte de español pese a ser igual de liviano.
Si `0.5B` resulta demasiado limitado en la evaluación (sección 5), el plan de
respaldo es `Qwen/Qwen2.5-1.5B-Instruct` — mismo tokenizer y familia, más capacidad.


In [29]:
# Importamos el modelo elegido directamente desde el Hub de Hugging Face.
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16).to(device)
print(f"Modelo cargado desde el Hub: {MODEL_ID}")


Loading weights: 100%|██████████| 290/290 [00:01<00:00, 175.41it/s]


Modelo cargado desde el Hub: Qwen/Qwen2.5-0.5B-Instruct


## 2 · Dataset: construcción, documentación y por qué es representativo

### Origen
Las **etiquetas** vienen del dataset público **PlantVillage** (Hughes & Salathé,
2015; Mohanty et al., 2016), que contiene 54,306 imágenes de hojas sanas y
enfermas en 38 clases sobre 14 especies de cultivo, organizadas como
`Cultivo___Enfermedad`. En esta fase **no usamos ninguna imagen**, solo la
taxonomía de labels.

No existe un dataset público de "recomendaciones agronómicas en español para las
etiquetas de PlantVillage", así que el contenido de cada recomendación fue
**curado por el equipo** a partir de fuentes primarias de extensión agrícola
universitaria (UMN Extension, Penn State, MSU, UF/IFAS, Cornell, UC IPM, etc.).
El detalle completo de fuente por clase está en `datos/base_conocimiento_plantvillage.json`
y las referencias agregadas en `datos/REFERENCIAS.md`.

### Por qué es representativo de la tarea
- Cubre las **38 clases oficiales** de PlantVillage (26 enfermedades/plagas + 12
  estados sanos), no un subconjunto arbitrario — el recomendador debe poder
  responder ante cualquier etiqueta que el futuro clasificador de imágenes prediga.
- Cada clase tiene entre 4 y 6 variantes de la misma pregunta (plantillas de
  instrucción distintas: "¿qué debo hacer si...", "dame recomendaciones para...",
  "detecté X, ¿qué acciones tomo?", etc.), para que el modelo generalice el
  **formato** de respuesta (identificación → acción → prevención) y no memorice
  una única redacción de la pregunta.
- Las respuestas siguen una estructura consistente en las tres partes que un
  agricultor necesita: qué es, qué hacer ahora, y cómo evitarlo a futuro.

### Limitaciones conocidas
- Las recomendaciones son generales/educativas, no reemplazan a un agrónomo en
  campo ni dan dosis exactas de agroquímico.
- Las fuentes primarias son mayoritariamente de EE. UU.; para uso en Colombia el
  manejo real (productos registrados, dosis) debería contrastarse con guías del
  ICA o gremios locales del cultivo.
- Es un dataset pequeño por clase (204 ejemplos / 38 clases ≈ 5-6 por clase),
  generado por *templating* sobre una base de conocimiento curada, no por
  recolección independiente de agricultores reales.


In [30]:
# Cargamos el dataset de fine-tuning ya construido en datos/
ruta_dataset = os.path.join(DATOS_DIR, "dataset_finetuning_plantvillage.jsonl")

ejemplos = []
with open(ruta_dataset, encoding="utf-8") as f:
    for linea in f:
        ejemplos.append(json.loads(linea))

print(f"{len(ejemplos)} ejemplos cargados desde {ruta_dataset}")

df_dataset = pd.DataFrame(ejemplos)
print(f"Clases distintas (labels de PlantVillage): {df_dataset['label_plantvillage'].nunique()} / 38")
df_dataset.sample(3, random_state=SEED)


204 ejemplos cargados desde datos\dataset_finetuning_plantvillage.jsonl
Clases distintas (labels de PlantVillage): 38 / 38


,label_plantvillage,cultivo,instruccion,respuesta
15,Potato___Early_blight,papa,¿Qué debo hacer si las hojas de mi papa tienen...,Se trata de tizón temprano en hojas de papa (A...
9,Tomato___Leaf_Mold,tomate,¿Qué debo hacer si las hojas de mi tomate tien...,Se trata de moho de la hoja en hojas de tomate...
115,Tomato___healthy,tomate,"No veo enfermedad en las hojas de mi tomate, ¿...",Se trata de sin enfermedad detectada en hojas ...


In [31]:
import json

# Mapeo: nombre que usamos nosotros -> nombre real de la carpeta en PlantVillage
correccion_labels = {
    "Corn___Cercospora_leaf_spot Gray_leaf_spot": "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
    "Corn___Common_rust": "Corn_(maize)___Common_rust_",
    "Corn___Northern_Leaf_Blight": "Corn_(maize)___Northern_Leaf_Blight",
    "Corn___healthy": "Corn_(maize)___healthy",
    "Cherry___Powdery_mildew": "Cherry_(including_sour)___Powdery_mildew",
    "Cherry___healthy": "Cherry_(including_sour)___healthy",
    "Pepper_bell___Bacterial_spot": "Pepper,_bell___Bacterial_spot",
    "Pepper_bell___healthy": "Pepper,_bell___healthy",
}

# 1) Corrige el DataFrame que ya está en memoria
df_dataset["label_plantvillage"] = df_dataset["label_plantvillage"].replace(correccion_labels)

# 2) Verifica que ya no queden diferencias contra las carpetas reales
clases_propias_corregidas = set(df_dataset["label_plantvillage"].unique())
print("Diferencias restantes:", clases_reales - clases_propias_corregidas or "ninguna ✅")

# 3) Corrige también el archivo JSON de la base de conocimiento en disco,
#    leyéndolo primero (no está en memoria, solo en datos/)
ruta_kb = os.path.join(DATOS_DIR, "base_conocimiento_plantvillage.json")
with open(ruta_kb, encoding="utf-8") as f:
    kb = json.load(f)

kb_corregida = {correccion_labels.get(k, k): v for k, v in kb.items()}

with open(ruta_kb, "w", encoding="utf-8") as f:
    json.dump(kb_corregida, f, ensure_ascii=False, indent=2)

# 4) Re-guarda también el .jsonl de fine-tuning ya corregido
df_dataset.to_json(
    os.path.join(DATOS_DIR, "dataset_finetuning_plantvillage.jsonl"),
    orient="records", lines=True, force_ascii=False
)

print("Archivos de datos/ corregidos y re-guardados.")

Diferencias restantes: ninguna ✅
Archivos de datos/ corregidos y re-guardados.


In [32]:
# Verificación cruzada contra el dataset REAL de PlantVillage, ya descargado
# localmente en datos/color/ (una subcarpeta por clase, formato estándar del
# dataset). Confirmamos que los nombres de clase de nuestra base de conocimiento
# coinciden exactamente con los nombres reales de las carpetas de imágenes.

ruta_imagenes = os.path.join(DATOS_DIR, "color")

if os.path.isdir(ruta_imagenes):
    clases_reales = {
        nombre for nombre in os.listdir(ruta_imagenes)
        if os.path.isdir(os.path.join(ruta_imagenes, nombre))
    }

    clases_propias = set(df_dataset["label_plantvillage"].unique())
    faltantes_en_propio = clases_reales - clases_propias
    extra_en_propio = clases_propias - clases_reales

    print(f"Clases encontradas en {ruta_imagenes}: {len(clases_reales)}")
    print(f"En PlantVillage real pero no en nuestra base: {faltantes_en_propio or 'ninguna'}")
    print(f"En nuestra base pero no vistas en las carpetas: {extra_en_propio or 'ninguna'}")

    # Bonus: cuántas imágenes hay por clase (útil para documentar el dataset de
    # imágenes también, de cara a la futura fase de computer vision)
    conteo_por_clase = {
        c: len(os.listdir(os.path.join(ruta_imagenes, c))) for c in sorted(clases_reales)
    }
    print(f"\\nTotal de imágenes: {sum(conteo_por_clase.values())}")
else:
    print(f"No se encontró la carpeta {ruta_imagenes}.")
    print("Esto no bloquea el resto del notebook, es solo una verificación de consistencia.")

Clases encontradas en datos\color: 38
En PlantVillage real pero no en nuestra base: ninguna
En nuestra base pero no vistas en las carpetas: ninguna
\nTotal de imágenes: 48370


In [33]:
# Split reproducible train/val/test, por grupo (label), con seed fija.
# No usamos train_test_split en cascada porque varias clases tienen pocos ejemplos
# (4 plantillas para las clases "healthy"), y un split estratificado en dos pasos
# puede dejar alguna clase con 1 solo miembro, lo que sklearn no puede dividir más.

rng = random.Random(SEED)
train_rows, val_rows, test_rows = [], [], []

for etiqueta, grupo in df_dataset.groupby("label_plantvillage"):
    indices = grupo.index.tolist()
    rng.shuffle(indices)
    n = len(indices)
    n_test = max(1, round(n * 0.15))
    n_val = max(1, round(n * 0.15))
    while n_test + n_val >= n:
        if n_val > 1:
            n_val -= 1
        elif n_test > 1:
            n_test -= 1
        else:
            break
    test_rows.extend(indices[:n_test])
    val_rows.extend(indices[n_test:n_test + n_val])
    train_rows.extend(indices[n_test + n_val:])

train_df = df_dataset.loc[train_rows].sample(frac=1, random_state=SEED).reset_index(drop=True)
val_df = df_dataset.loc[val_rows].sample(frac=1, random_state=SEED).reset_index(drop=True)
test_df = df_dataset.loc[test_rows].sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"Clases en train: {train_df['label_plantvillage'].nunique()}/38 | "
      f"val: {val_df['label_plantvillage'].nunique()}/38 | "
      f"test: {test_df['label_plantvillage'].nunique()}/38")


Train: 128 | Val: 38 | Test: 38
Clases en train: 38/38 | val: 38/38 | test: 38/38


---
## Lab A · El baseline

**La pieza más importante de M1.** Antes de entrenar, medimos qué tan bien hace el
modelo la tarea *sin* fine-tuning. Este número es el punto de comparación: sin él
no se puede demostrar que el fine-tuning sirvió.


In [34]:
def preguntar(modelo, prompt, max_new=120):
    """Le da un prompt al modelo y devuelve su respuesta."""
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = modelo.generate(**inputs, max_new_tokens=max_new,
                               do_sample=False, pad_token_id=tokenizer.pad_token_id)
    texto = tokenizer.decode(out[0], skip_special_tokens=True)
    return texto[len(prompt):].strip()

# Pregunta de referencia del equipo (queda fija para comparar baseline vs. después)
pregunta = "¿Qué debo hacer si las hojas de mi papa tienen tizón tardío?"

prompt_formateado = f"Pregunta: {pregunta}\nRespuesta:"

print("PREGUNTA:", pregunta)
respuesta_baseline = preguntar(model, prompt_formateado)
print("BASELINE:", respuesta_baseline)


PREGUNTA: ¿Qué debo hacer si las hojas de mi papa tienen tizón tardío?
BASELINE: Si las hojas de tu papá tienen tizón tardío, puedes seguir estos pasos para ayudar a prevenir o tratar el problema:

1. Consulta con un profesional de la salud: Un médico puede diagnosticar el problema y proporcionar una recomendación médica.

2. Mantén una dieta equilibrada: Comprueba que tus alimentos sean balanceados y no contienen demasiado azúcar o grasas saturadas.

3. Evita los alimentos procesados: Los procesados pueden aumentar el riesgo de tizones en algunas personas.

4


In [35]:
# Baseline cuantitativo: además del ejemplo puntual de arriba, medimos ROUGE-L
# del modelo SIN fine-tuning contra las respuestas de referencia en todo test_df.
# Esto da un número (no solo una lectura cualitativa) para sustentar "mejoró".

import evaluate

rouge = evaluate.load("rouge")

def evaluar_modelo(modelo, df_eval, nombre="modelo"):
    predicciones, referencias = [], []
    for _, fila in df_eval.iterrows():
        prompt = f"Pregunta: {fila['instruccion']}\nRespuesta:"
        pred = preguntar(modelo, prompt)
        predicciones.append(pred)
        referencias.append(fila["respuesta"])
    resultado = rouge.compute(predictions=predicciones, references=referencias)
    resultado["nombre"] = nombre
    return resultado, predicciones

resultados_baseline, preds_baseline = evaluar_modelo(model, test_df, nombre="baseline_zero_shot")
resultados_baseline


{'rouge1': np.float64(0.25404991012442524),
 'rouge2': np.float64(0.04529758496037638),
 'rougeL': np.float64(0.14807510083197847),
 'rougeLsum': np.float64(0.17132998901083954),
 'nombre': 'baseline_zero_shot'}

---
## 3 · Sus datos, en formato de entrenamiento

Convertimos los ejemplos al formato que el modelo espera: un solo texto por
ejemplo, con la pregunta y la respuesta juntas.


In [36]:
from datasets import Dataset

def formatear(ej):
    texto = f"Pregunta: {ej['instruccion']}\nRespuesta: {ej['respuesta']}{tokenizer.eos_token}"
    return {"text": texto}

train_dataset = Dataset.from_list([formatear(e) for e in train_df.to_dict("records")])
val_dataset = Dataset.from_list([formatear(e) for e in val_df.to_dict("records")])

def tokenizar(batch):
    out = tokenizer(batch["text"], truncation=True, max_length=256, padding="max_length")
    # En Causal LM, las etiquetas (labels) son una copia de los input_ids.
    out["labels"] = out["input_ids"].copy()
    return out

train_dataset = train_dataset.map(tokenizar, remove_columns=["text"])
val_dataset = val_dataset.map(tokenizar, remove_columns=["text"])
print("Dataset de entrenamiento listo:", train_dataset)


Map: 100%|██████████| 38/38 [00:00<00:00, 2214.52 examples/s]

Dataset de entrenamiento listo: Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 128
})


---
## Lab B · Fine-tuning con LoRA

Configuramos LoRA con la librería `peft`. Usamos `target_modules` un poco más
amplio que el mínimo de clase (`q_proj`, `v_proj`) porque nuestro dataset cubre
38 clases distintas (mucho más variado que el ejercicio de 20 ejemplos de un solo
dominio del laboratorio), y cubrir más proyecciones de atención ayuda al modelo a
diferenciar mejor entre etiquetas.


In [37]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,                # rank: un poco más alto que el default de clase (8) por
                          # la mayor variedad de clases/vocabulario del dominio
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()   # deben ver <1% entrenable


trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359


In [38]:
from transformers import Trainer, TrainingArguments

args = TrainingArguments(
    output_dir="./lora-out",
    learning_rate=2e-4,
    num_train_epochs=6,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    eval_strategy="epoch",
    logging_steps=10,
    report_to="none",
    fp16=(device == "cuda"),
    seed=SEED,
)

trainer = Trainer(model=model, args=args, train_dataset=train_dataset, eval_dataset=val_dataset)
trainer.train()
print("Entrenamiento terminado.")


Epoch,Training Loss,Validation Loss
1,4.476087,2.014367
2,1.877933,1.544437
3,1.675298,1.277164
4,1.323791,1.104220
5,1.142474,0.991915
6,1.059000,0.946886


Entrenamiento terminado.


---
## Lab C · ¿Mejoró?

Le hacemos la **misma pregunta** del baseline al modelo ya entrenado, y comparamos.


In [39]:
model.eval()

print("PREGUNTA:", pregunta)
print()
print("ANTES (baseline):")
print(respuesta_baseline)
print()
respuesta_finetuned = preguntar(model, prompt_formateado)
print("DESPUÉS del fine-tuning:")
print(respuesta_finetuned)


PREGUNTA: ¿Qué debo hacer si las hojas de mi papa tienen tizón tardío?

ANTES (baseline):
Si las hojas de tu papá tienen tizón tardío, puedes seguir estos pasos para ayudar a prevenir o tratar el problema:

1. Consulta con un profesional de la salud: Un médico puede diagnosticar el problema y proporcionar una recomendación médica.

2. Mantén una dieta equilibrada: Comprueba que tus alimentos sean balanceados y no contienen demasiado azúcar o grasas saturadas.

3. Evita los alimentos procesados: Los procesados pueden aumentar el riesgo de tizones en algunas personas.

4

DESPUÉS del fine-tuning:
Se trata de tizón tardío en hojas de papa (Phytophthora infestans). Identificación: lesiones menores de 10% del diámetro de la hoja, con pequeñas manchas o curvas rojizas que se expanden hasta cubrir el diametro de la hoja; es más frecuente en frutos ya maduros. Acción recomendada: no existe cura para papa, ya que es una enfermedad muy severa y puede causar defolacinación y mortalidad del fruto;

In [40]:
# Comparación cuantitativa: mismo test set, misma métrica que el baseline.
resultados_ft, preds_ft = evaluar_modelo(model, test_df, nombre="fine_tuned_lora")

comparacion = pd.DataFrame([resultados_baseline, resultados_ft]).set_index("nombre")
comparacion


,rouge1,rouge2,rougeL,rougeLsum
nombre,,,,
baseline_zero_shot,0.254050,0.045298,0.148075,0.171330
fine_tuned_lora,0.452909,0.230517,0.344384,0.343938


In [ ]:
# Inspección cualitativa lado a lado, incluyendo casos donde el fine-tuning
# no haya mejorado tanto.
comparacion_ejemplos = test_df.reset_index(drop=True).copy()
comparacion_ejemplos["prediccion_baseline"] = preds_baseline
comparacion_ejemplos["prediccion_fine_tuned"] = preds_ft
comparacion_ejemplos[["instruccion", "respuesta", "prediccion_baseline", "prediccion_fine_tuned"]].head(8)


,instruccion,respuesta,prediccion_baseline,prediccion_fine_tuned
0,¿Qué debo hacer si las hojas de mi tomate tien...,Se trata de infestación por ácaros/araña roja ...,Si tienes hojas de tu tomate con infestaciones...,Se trata de infestación por ácaros/araña roja ...
1,Dame recomendaciones para tratar virus del mos...,Se trata de virus del mosaico del tomate en ho...,El mosaico del tomate es un tipo de virus que ...,Se trata de virus del mosaico del tomate (Toma...
2,"Las hojas de mi arándano se ven sanas, ¿qué de...",Se trata de sin enfermedad detectada en hojas ...,"Para mantener las hojas sanas y brillantes, pu...",Se trata de hojas de arándano sin síntomas. Id...
3,Detecté tizón foliar (mancha de Isariopsis) en...,Se trata de tizón foliar (mancha de Isariopsis...,La mancha de Isariopsis es una señal de enferm...,Se trata de tizón foliar (mancha de Isariopsis...
4,Explícame cómo manejar tizón tardío en hojas d...,Se trata de tizón tardío en hojas de tomate (P...,Tizón tardío es un problema común que afecta a...,Se trata de tizón tardío en hojas de tomate (T...
5,¿Qué debo hacer si las hojas de mi fresa (frut...,Se trata de quemazón de la hoja (leaf scorch) ...,Si las hojas de tu fruta (frutilla) tienen que...,Se trata de quemazón de la hoja (leaf scorch) ...
6,¿Qué recomendaciones generales me das para cui...,Se trata de sin enfermedad detectada en hojas ...,"Para cuidar un cultivo sano de cerezo, aquí ti...",Se trata de información general sobre cultivo ...
7,No veo enfermedad en las hojas de mi fresa (fr...,Se trata de sin enfermedad detectada en hojas ...,La enfermedad que se puede prevenir es la infe...,Se trata de no ver enfermedad en hojas de fres...


## 4 · Guardar el modelo

In [ ]:
model.save_pretrained("./mi-modelo-lora")
tokenizer.save_pretrained("./mi-modelo-lora")
print("Guardado. El adaptador LoRA pesa solo unos MB — esa es la gracia de LoRA.")

# Para subirlo al Hub (opcional):
# model.push_to_hub("su-usuario/su-modelo")


---
## 5 · Conclusiones y limitaciones

- Se seleccionó y justificó `Qwen2.5-0.5B-Instruct` con evidencia de tamaño,
  licencia, idioma y comportamiento del tokenizador (sección 1).
- Se documentó el origen, la construcción y las limitaciones del dataset, además
  de por qué los 204 ejemplos son representativos de la tarea (sección 2), con
  fuentes completas en `datos/REFERENCIAS.md`.
- Se midió un baseline explícito (Lab A) y se comparó contra el modelo fine-tuned
  con la misma pregunta y las mismas métricas sobre el test set (Lab C).
- El fine-tuning se hizo con LoRA de forma reproducible (seed fija, hiperparámetros
  documentados en la celda de configuración).

**Trabajo futuro (fuera de alcance de M1):**
- Extender la base de conocimiento con fuentes específicas de Colombia/ICA.
- Integrar el clasificador de imágenes (computer vision) que prediga la etiqueta
  de PlantVillage a partir de una foto de la hoja, y conectarlo con este módulo
  de recomendación de texto.
- Evaluar con métricas adicionales a ROUGE (evaluación humana o LLM-as-judge),
  ya que ROUGE no captura bien la corrección agronómica del contenido.
